# 05 Testimony filter (Llama 3.1 8B, Ollama)

Runs locally with `llama3.1:8b` via Ollama. Needs ollama running (`ollama serve`) and the model pulled (`ollama pull llama3.1:8b`).

## Settings

In [ ]:
import json, re, time
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm

# input file and output folder (change these for each subreddit)
INPUT_FILE = Path("path/to/datapoints.csv")
OUT_DIR    = Path("path/to/output_folder")
TEXT_COL   = "datapoint"

# model settings
MODEL       = "llama3.1:8b"
OLLAMA_URL  = "http://localhost:11434/api/generate"
TEMPERATURE = 0.0
NUM_CTX     = 4096     # context window
MAX_CHARS   = 6000     # cut texts to this length (~1500 tokens)
MAX_TOKENS  = 50       # answer is one short json
MAX_RETRIES, RETRY_DELAY = 3, 5
RECALL_N, SEED = 100, 42

## Prompt

In [ ]:
# testimony definition (it is the same for both subreddits)
TESTIMONY_DEFINITION = (
    "The text contains a first-person account in which the author describes what led them to their "
    "political views, involvement, or identity. This includes descriptions of influences, experiences, "
    "or reasons that shaped the author's political perspective - even if the connection to a specific "
    "movement or identity is implicit rather than explicitly stated. Brief causal statements also qualify "
    '(e.g., "It was reading one particular book", "It was a conversation with a close friend"). '
    "The testimony may be embedded within a longer text that also contains advice, questions, or "
    "discussion. Do NOT require the author to explicitly name a movement or a political label."
)

def build_prompt(text):
    # full prompt sent to llama
    return (
        "You are a research assistant helping to filter Reddit data for a social science study.\n\n"
        "Your task is to determine whether a Reddit post or comment contains a TESTIMONY.\n\n"
        f"DEFINITION OF TESTIMONY:\n{TESTIMONY_DEFINITION}\n\n"
        f'TEXT:\n"""\n{text}\n"""\n\n'
        'Respond ONLY with valid JSON in this exact format:\n{"is_testimony": true or false}'
    )

# print to check what the prompt looks like
print(build_prompt("[EXAMPLE TEXT]"))

## Model call
`classify` gives `True` / `False`, or `None` if there was no valid answer.

In [ ]:
def classify(text):
    # send to ollama, read the json answer, retry if it fails
    payload = {"model": MODEL, "prompt": build_prompt(text[:MAX_CHARS]), "stream": False,
               "options": {"temperature": TEMPERATURE, "num_ctx": NUM_CTX, "num_predict": MAX_TOKENS}}
    for attempt in range(MAX_RETRIES):
        try:
            raw = requests.post(OLLAMA_URL, json=payload, timeout=120).json()["response"]
            m = re.search(r"\{.*?\}", raw, re.DOTALL)
            value = json.loads(m.group()).get("is_testimony") if m else None
            if isinstance(value, bool):
                return value
        except Exception as e:
            print(f"  attempt {attempt + 1} failed: {e}")
        time.sleep(RETRY_DELAY)
    return None

# quick test that ollama is running
print("Connection test:", classify("I became involved because of how I saw people treated at my workplace."))

## Run
Answers go into `_progress.csv` as they come in. When everything is done they are merged into the datapoints
(`llama_filtered.csv`), the recall sample is saved (`recall_sample_TO_CODE.csv`) and the progress file is deleted.

In [ ]:
df = pd.read_csv(INPUT_FILE, low_memory=False)
texts = df[TEXT_COL].fillna("").astype(str)
OUT_DIR.mkdir(parents=True, exist_ok=True)
progress = OUT_DIR / "_progress.csv"

# classify all rows that are not done yet, save each answer right away
done = set(pd.read_csv(progress)["row"]) if progress.exists() else set()
with open(progress, "a") as f:
    if not done:
        f.write("row,is_testimony\n")
    for i in tqdm([i for i in range(len(df)) if i not in done]):
        f.write(f"{i},{classify(texts[i])}\n")
        f.flush()

# merge answers back into the datapoints and save
res = pd.read_csv(progress).drop_duplicates("row").set_index("row")["is_testimony"]
df["is_testimony"] = res.reindex(range(len(df))).map({True: True, False: False, "True": True, "False": False}).values
df.to_csv(OUT_DIR / "llama_filtered.csv", index=False)

# recall sample from the negatives
negatives = df[df["is_testimony"] == False]
sample = negatives.sample(min(RECALL_N, len(negatives)), random_state=SEED)
cols = [c for c in ["datapoint_id", "id", "datapoint_type", TEXT_COL] if c in df.columns]
sample[cols].assign(manual_is_testimony="").to_csv(OUT_DIR / "recall_sample_TO_CODE.csv", index=False)
progress.unlink()

## Summary
Saved as `llama_filter_counts.csv`. The `manual_is_testimony` column (1/0) in the recall sample is coded by hand.

In [ ]:
# counts: testimony / not testimony / failed (no valid answer)
counts = pd.Series({"datapoints": len(df),
                    "testimony": int((df["is_testimony"] == True).sum()),
                    "not testimony": len(negatives),
                    "failed": int(df["is_testimony"].isna().sum())})
display(counts)
counts.to_csv(OUT_DIR / "llama_filter_counts.csv", header=["n"])